# Notebook 03 — Semantic-local attention: bounded promise test

**Run All prepares and runs the real experiment.** Three fresh models share the encoder scaffold, predictor, training images, augmentations, masks, optimizer and successful-update budget: ViT attention, original FMM, and normalized semantic summaries with exact nearby interactions.

Inputs: the existing audited Food101 10,100-image training / 2,020-image development split. Official test images remain reserved. Outputs: `artifacts/notebook_03/<RUN_ID>/`, frozen-feature development probes, compute measurements and a report. Checkpoints: `checkpoints/<RUN_ID>/{vit,fmm,semantic}/` and probes. Calibration chooses at most 1,500 updates per model using timing only. Target runtime about two hours; calibration, benchmark, preflight, training and evaluation share an absolute three-hour wall-clock cap.

This is one paired seed and a short learning budget. It can identify promising quality/cost behavior; it cannot settle statistical superiority or asymptotic scaling.


## 1. Import definitions only
Use the fresh `Python (ahr-go)` kernel. Only explicitly tagged `ahr-export` cells are imported.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)

from fmm_jepa.semantic_pilot import read_config, prepare_semantic_pilot, launch_semantic_pilot, wait_for_semantic_pilot, artifact_path


## 2. Select a run and inspect the requested configuration
Keep RUN_ID unchanged to resume/reconnect. A changed source/configuration requires a new run identity. An exhausted wall-clock budget is never automatically extended.

In [ ]:
RUN_ID = 'semantic-local-pilot-002'
CONFIG = read_config(ROOT)
CONFIG

## 3. Calibrate, freeze choices and certify exact recovery
Checks selected source-image hashes, measures real-image updates, fixes the common budget, benchmarks all mixers, and verifies four uninterrupted updates equal checkpoint-restored two plus two for every model. Calibration weights are discarded.

In [ ]:
PREPARED = prepare_semantic_pilot(ROOT, RUN_ID, CONFIG)
CONFIG = PREPARED['config']
PREPARED

## 4. Launch or resume in detached tmux

**Exact resume:** restart the same project kernel, leave RUN_ID unchanged, and click Run All. This cell (`launch-detached-semantic-pilot`) reconnects to an active worker or resumes its latest validated full-state checkpoints. Invalid checksums, changed definitions/configuration, or an exhausted deadline stop explicitly. Full state includes model/teacher, optimizer, scheduler, RNG, sampler cursor and metrics. Keep one writer per run.

Attach with the command printed below; detach with **Ctrl-b, then d**. Interrupting the following monitor leaves training running.

In [ ]:
LAUNCH = launch_semantic_pilot(ROOT, RUN_ID)
LAUNCH

## 5. Wait for the saved results
Training advances in equal 100-update rounds and reserves time for six initialized/trained frozen-feature probes. The worker retains latest two valid training checkpoints and the best recovery checkpoint; primary comparison uses the common final budget.

In [ ]:
COMPLETION = wait_for_semantic_pilot(ROOT, RUN_ID)
COMPLETION

## 6. Read the evidence
Inspect measured learning, uncertainty and quality against compute. All claims must be based on completed saved stages; incomplete comparisons remain inconclusive.

In [ ]:
from IPython.display import display, Markdown
ARTIFACT = artifact_path(ROOT, RUN_ID)
reports = sorted(ARTIFACT.glob('*results*.md'))
if not reports:
    raise RuntimeError('No completed results report found; inspect pilot_status.json and the worker log.')
for report in reports:
    display(Markdown(report.read_text()))